# Predicting beer review ratings
**Team members:** Add your names before submission.

We estimate `review/overall` using ABV, component review scores (appearance,
aroma, palate, taste), reviewer age at review time, missing-age indicator,
review date, beer style and gender. Brewery ID is excluded. Because component
scores come from the same review, this estimates an overall score from available
review information; it does not predict preferences before a review exists.

We compare a training-mean baseline, a learned linear baseline, and two sigmoid
networks with hidden layers **16 → 16**: one linear regression output and eleven
softmax rating probabilities on the grid **0, 0.5, …, 5**. The probability mean,
mode, and median are different decision rules from the same trained network.

Run cells in order with **Runtime → Run all**. CPU is sufficient; these NumPy/SciPy
models do not use a Colab GPU. First execution may prompt for the dataset upload.


## 1. Get the code from GitHub
This cell clones the repository on the first run and pulls updates on later runs.
It refuses to discard local edits or merge divergent changes. After pulling code
updates, restart the runtime and run all cells. Updating this checkout does not
update the notebook already open in your browser: reopen the GitHub notebook
to get new notebook cells.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/Dantsel04/Beer_ratings.git"
CHECKOUT = Path("/content/Beer_ratings")
if CHECKOUT.exists():
    if not (CHECKOUT / ".git").exists():
        raise RuntimeError(f"{CHECKOUT} exists but is not a Git checkout")
    changes = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "status", "--porcelain"], text=True)
    if changes.strip():
        raise RuntimeError("Save your local code edits before pulling updates.")
    subprocess.run(["git", "-C", str(CHECKOUT), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(CHECKOUT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(CHECKOUT / "requirements.txt")], check=True)
if str(CHECKOUT) not in sys.path:
    sys.path.insert(0, str(CHECKOUT))
print("Code commit:", subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip())


## 2. Load the original dataset
All 50,000 records are retained. The `.json` file actually contains one Python
dictionary literal per line; the loader uses `ast.literal_eval`, never `eval`.
The SHA-256 check ensures we use the exact dataset from the existing experiments.

The course server may reject automated downloads. If prompted, upload your local
`beer_50000.json`. You can instead set `DATA_PATH` to a file already in the runtime
or mounted Google Drive. Data in `/content` disappears when Colab deletes a runtime.


In [ ]:
import hashlib
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from beer_models import data, regression, probability

DATA_PATH = Path("/content/data/beer_50000.json")  # Change for a file in Drive if desired.
DATA_SHA256 = "8cd4f0958b6bff5de7b072b8fdc6eef41b7a7c2342cc7d385956ccfd862c70ab"
if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    url = "https://www.ece.villanova.edu/~xjiao/course/ECE5400/dataset/beer_50000.json"
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=30) as response:
            payload = response.read()
        if hashlib.sha256(payload).hexdigest() != DATA_SHA256:
            raise ValueError("Downloaded content does not match the original dataset")
        DATA_PATH.write_bytes(payload)
    except Exception as error:
        print("Course download unavailable:", error)
        print("Select your original beer_50000.json file.")
        from google.colab import files
        uploaded = files.upload()
        if "beer_50000.json" not in uploaded:
            raise ValueError("Please upload beer_50000.json")
        DATA_PATH.write_bytes(uploaded["beer_50000.json"])
        del uploaded
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest() == DATA_SHA256, "Dataset mismatch"
records = data.read_records(DATA_PATH)
numeric, cats, y = data.raw_arrays(records)
assert len(y) == 50000
probability.labels(y)  # Check every target belongs to the half-point grid.
print(f"Records: {len(y):,}; zero targets: {(y == 0).sum()}; missing ages: {np.isnan(numeric[:, 5]).sum():,}")
display(pd.Series(y).value_counts().sort_index().rename("Review count").to_frame())


## 3. Reproduce the split and fit preprocessing
The fixed seed reproduces the original record indices: 32,000 fitting, 8,000
validation, and 10,000 test. Together fitting and validation form the 80%
development set. Median age imputation, scaling, and category vocabularies are
fitted on fitting records only during selection. Unseen categories have a separate column.

We also check exact duplicate records across the boundaries. Beers and reviewers
can still occur in multiple partitions: this evaluates random held-out reviews,
not generalization to entirely new beers or reviewers.


In [ ]:
order = np.random.default_rng(7428).permutation(len(y))
test, development = order[:10000], order[10000:]
validation, fitting = development[:8000], development[8000:]
assert len(set(fitting) | set(validation) | set(test)) == len(y)
assert not set(development).intersection(test)
import json
fingerprints = [hashlib.sha256(json.dumps(r, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
                for r in records]
for left, right in [(fitting, validation), (development, test)]:
    assert not ({fingerprints[i] for i in left} & {fingerprints[i] for i in right}), "Duplicate record crosses split"
preprocessor = data.fit_preprocessor(numeric, cats, fitting)
x = data.transform(numeric, cats, preprocessor, with_brewery=False)
print("Fitting / validation / test:", len(fitting), len(validation), len(test))
print("Selection input width:", x.shape[1])


## 4. Select training settings on validation data
The model implementation is in the three small `beer_models` modules in GitHub.
It preserves the original manual backpropagation: sigmoid hidden activations,
normal(0, 0.1) initial weights, zero biases, mini-batches of 512, and either SGD
with momentum or Adam. There is no dropout or weight decay.

Default **full** mode compares three optimizer recipes and three seeds per network
(18 selection fits total). Each fit allows 200 epochs, validates every five epochs,
and uses patience 35. Regression checkpoints minimize validation MAE; softmax
checkpoints minimize validation negative log likelihood (cross-entropy).

**Quick** mode is a pipeline demonstration using one recipe, one seed, and 20 epochs.
Its scores are not the full experiment results. Architecture and feature choices
come from earlier exploratory work on this dataset; this is a focused rerun, not
a new unbiased architecture search.


In [ ]:
MODE = "full"  # Use "quick" to check the entire workflow before a full run.
assert MODE in {"full", "quick"}
LAYOUT = [16, 16]
SEEDS = [11, 22, 33] if MODE == "full" else [11]
CONFIGS = [
    dict(name="minibatch_momentum", optimizer="sgd", lr=.03, momentum=.9,
         batch_size=512, max_epochs=200, patience=35),
    dict(name="adam_0.001", optimizer="adam", lr=.001, momentum=0.,
         batch_size=512, max_epochs=200, patience=35),
    dict(name="adam_0.003", optimizer="adam", lr=.003, momentum=0.,
         batch_size=512, max_epochs=200, patience=35),
]
if MODE == "quick":
    CONFIGS = [dict(CONFIGS[-1], max_epochs=20, patience=10)]

selection_rows, histories = [], {}
with threadpool_limits(limits=1):
    for family, module, criterion in [
        ("regression", regression, "val_mae"), ("softmax", probability, "val_nll")]:
        for config in CONFIGS:
            for seed in SEEDS:
                weights, info = module.train(
                    x[fitting], y[fitting], x.shape[1], LAYOUT, config, seed,
                    x[validation], y[validation])
                value = min(row[criterion] for row in info["history"])
                selection_rows.append(dict(family=family, recipe=config["name"], seed=seed,
                                           validation_score=value, best_epoch=info["best_epoch"]))
                histories[(family, config["name"], seed)] = info["history"]
                print(f"{family}, {config['name']}, seed {seed}: {criterion}={value:.6f}, epoch={info['best_epoch']}")
selection = pd.DataFrame(selection_rows)
summary = selection.groupby(["family", "recipe"]).agg(
    mean_validation_score=("validation_score", "mean"),
    seed_sd=("validation_score", "std"), median_epoch=("best_epoch", "median")).reset_index()
display(summary)
selected = {}
for family in ["regression", "softmax"]:
    winner = summary[summary.family == family].sort_values(
        ["mean_validation_score", "recipe"]).iloc[0]
    selected[family] = dict(recipe=winner.recipe, epochs=int(np.rint(winner.median_epoch)))
print("Frozen choices:", selected)


## 5. Refit on all 40,000 development reviews
Refit preprocessing on development records. Initialize fresh networks with fixed
seed 11 and train for the median selected checkpoint epoch. The test set does
not determine training settings or refit duration. The linear baseline uses the
same encoded inputs and an intercept, fitted by sparse least squares.


In [ ]:
from scipy import sparse
from scipy.sparse.linalg import lsqr

final_preprocessor = data.fit_preprocessor(numeric, cats, development)
xf = data.transform(numeric, cats, final_preprocessor, with_brewery=False)
models = {}
with threadpool_limits(limits=1):
    for family, module in [("regression", regression), ("softmax", probability)]:
        choice = selected[family]
        config = next(c for c in CONFIGS if c["name"] == choice["recipe"])
        models[family], _ = module.train(xf[development], y[development], xf.shape[1],
                                         LAYOUT, config, 11, epochs=choice["epochs"])
    linear_x = sparse.hstack([xf, np.ones((len(y), 1))], format="csr", dtype=np.float64)
    linear_fit = lsqr(linear_x[development], y[development], atol=1e-9, btol=1e-9, iter_lim=5000)
    if linear_fit[1] not in {0, 1, 2, 4, 5}:
        raise RuntimeError(f"Linear least-squares solver did not converge: stop code {linear_fit[1]}")
    linear_weights = linear_fit[0]
print("Refits complete. Final input width:", xf.shape[1])


## 6. Evaluate the frozen models
MAE is the average absolute rating error. MAPE is the average absolute percentage
error, expressed as a percentage. Ordinary MAPE is undefined for zero targets;
the table reports it only when all test targets are nonzero, and separately reports
MAPE on nonzero targets with their count. No zero-target review is removed from
training or from MAE.

Rounded regression uses nearest-half-point rounding with NumPy ties-to-even,
then clipping to [0, 5]. Softmax mode selects the most likely score (lower score
on ties); median selects the first score whose cumulative probability reaches 0.5;
mean is the probability-weighted score. All are reported, rather than selecting a
decoder from test performance. Rounding itself can improve MAE on discrete labels.


In [ ]:
actual = y[test]
raw = regression.forward(xf[test], models["regression"])[0]
p, logp, _ = probability.forward(xf[test], models["softmax"])
assert np.isfinite(p).all()
np.testing.assert_allclose(p.sum(axis=1), 1, atol=1e-6)
predictions = {
    "Training mean": np.full(len(test), y[development].mean()),
    "Linear regression": np.asarray(linear_x[test] @ linear_weights).ravel(),
    "Neural regression": raw,
    "Neural regression (rounded)": np.clip(np.rint(raw * 2) / 2, 0, 5),
    **{f"Softmax ({rule})": values for rule, values in probability.decode(p).items()},
}
rows = []
for name, prediction in predictions.items():
    metrics = data.metrics(actual, prediction)
    rows.append(dict(model=name, MAE=metrics["mae"], RMSE=metrics["rmse"],
                     MAPE_percent=metrics["ordinary_mape_percent"],
                     nonzero_MAPE_percent=metrics["nonzero_target_mape_percent"],
                     MAPE_nonzero_count=int((actual != 0).sum()),
                     zero_targets=metrics["zero_target_count"],
                     within_half_percent=100 * np.mean(abs(prediction - actual) <= .5),
                     exact_percent=100 * np.mean(prediction == actual)))
results = pd.DataFrame(rows).set_index("model")
display(results.round(4))
print("Run mode:", MODE)
print("Softmax test NLL:", probability.metrics(actual, p, logp)["nll"])


## 7. Inspect training and predictions
Training curves below come from seed 11 of the selected recipes during selection,
not the development refits. Test comparisons are exploratory: this held-out set
was inspected in earlier project experiments. Small differences do not establish
that one model is reliably better. The newly added linear baseline has no historical
score; its result is calculated by this notebook.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, family, metric in zip(axes, ["regression", "softmax"], ["mae", "nll"]):
    history = pd.DataFrame(histories[(family, selected[family]["recipe"], 11)])
    ax.plot(history.epoch, history[f"train_{metric}"], label="Fitting")
    ax.plot(history.epoch, history[f"val_{metric}"], label="Validation")
    ax.set(title=f"{family}: selection curves", xlabel="Epoch", ylabel=metric.upper())
    ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hexbin(actual, raw, gridsize=30, mincnt=1, cmap="viridis")
axes[0].plot([0, 5], [0, 5], "r--")
axes[0].set(xlabel="Actual rating", ylabel="Predicted rating", title="Continuous neural regression")
for name in ["Neural regression", "Neural regression (rounded)", "Softmax (median)"]:
    errors = np.sort(abs(predictions[name] - actual))
    axes[1].plot(errors, np.arange(1, len(errors) + 1) / len(errors), label=name)
axes[1].set(xlabel="Absolute error", ylabel="Fraction within error", xlim=(0, 1.5))
axes[1].legend()
plt.tight_layout()
plt.show()

examples = pd.DataFrame({"record_index": test[:10], "actual": actual[:10],
                         **{name: value[:10] for name, value in predictions.items()}})
display(examples.round(3))
plt.bar(probability.SCORES, p[0], width=.35)
plt.xticks(probability.SCORES)
plt.xlabel("Rating")
plt.ylabel("Predicted probability")
plt.title(f"First test review: actual rating {actual[0]:g}")
plt.show()


## 8. Save the run
Save results, exact split indices, training choices, preprocessing, learned weights,
and predictions with the Git commit and data hash. Download the ZIP before the
runtime expires. Historical Intel GPU refits can differ slightly from this CPU rerun.


In [ ]:
import shutil
from datetime import datetime, timezone

OUTPUT = Path("/content/beer_rating_outputs") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT.mkdir(parents=True, exist_ok=False)
results.to_csv(OUTPUT / "metrics.csv")
selection.to_csv(OUTPUT / "validation_runs.csv", index=False)
pd.DataFrame({"record_index": test, "actual": actual, **predictions}).to_csv(
    OUTPUT / "test_predictions.csv", index=False)
np.savez_compressed(OUTPUT / "splits.npz", fitting=fitting, validation=validation,
                    development=development, test=test)
np.savez_compressed(OUTPUT / "softmax_probabilities.npz", scores=probability.SCORES, probabilities=p)
for name, weights in models.items():
    np.savez_compressed(OUTPUT / f"{name}_weights.npz", **weights)
np.save(OUTPUT / "linear_weights.npy", linear_weights)
manifest = dict(mode=MODE, selected=selected, layout=LAYOUT, seeds=SEEDS, configs=CONFIGS,
                final_seed=11, with_brewery=False, data_sha256=DATA_SHA256,
                code_commit=subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip(),
                python=sys.version, numpy=np.__version__, final_preprocessor=final_preprocessor,
                histories={"/".join(map(str, key)): value for key, value in histories.items()},
                interpretation="Exploratory comparisons on the previously inspected test split")
(OUTPUT / "run.json").write_text(json.dumps(manifest, indent=2, allow_nan=False))
archive = shutil.make_archive(str(OUTPUT), "zip", OUTPUT)
print("Saved:", archive)
from google.colab import files
files.download(archive)


## Submission checklist
- Add every team member's name above and to the report.
- Use full mode for reported final results; preserve the downloaded run ZIP.
- Explain the target, inputs, two network outputs, MAE and MAPE in the 1–2 page report.
- Include the rounded-regression comparator when discussing probability outputs.
- Put the Colab notebook link at the **beginning** of the report and enable sharing
  so a third party can view it. Test the link in a signed-out browser.
